# 02 — Baseline model

**Goal:** Train the dumbest models that beat random, so we have a benchmark to improve on later.

**Outputs of this notebook:**
- A serialized `joblib` model file in `models/`
- A JSON file of metrics for later comparison
- A precision-recall curve plot
- Calibrated decision threshold (not the default 0.5)

**Why this notebook is critical:** every later model — feature-engineered XGBoost, retrained models from Prefect — gets compared against this baseline. If you can't establish the baseline cleanly, you have no way to claim improvement.

In [ ]:
import os

print("MLFLOW_TRACKING_URI:", os.environ.get("MLFLOW_TRACKING_URI"))
print("AWS_DEFAULT_REGION:", os.environ.get("AWS_DEFAULT_REGION"))
print("MLFLOW_ARTIFACT_ROOT:", os.environ.get("MLFLOW_ARTIFACT_ROOT"))

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import logging
from datetime import datetime

import joblib
import matplotlib.pyplot as plt

# NEW
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from fraud_mlops.config import FIGURES_DIR, MODELS_DIR, RANDOM_SEED, ensure_dirs
from fraud_mlops.data import load_paysim, split_features_label, time_based_split
from fraud_mlops.metrics import (
    evaluate_at_threshold,
    find_threshold_for_precision,
)

# NEW
from fraud_mlops.tracking import setup_mlflow

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
ensure_dirs()
np.random.seed(RANDOM_SEED)

# NEW — initialize MLflow before any modeling
setup_mlflow(experiment_name="fraud_baseline")

## Load data and split by time

In [ ]:
# This time, drop_leaky=True — we no longer need to see the leaky columns.
df = load_paysim(drop_leaky=True)

# Time-based split. The last 20% of time becomes test.
train_df, test_df = time_based_split(df, test_fraction=0.2)

X_train, y_train = split_features_label(train_df)
X_test, y_test = split_features_label(test_df)

print(f"Train: {len(X_train):,}  ({y_train.mean():.4%} fraud)")
print(f"Test:  {len(X_test):,}  ({y_test.mean():.4%} fraud)")

## Trivial baseline: predict majority class

Every metric we compute should beat this by enough to justify the model's existence.

In [ ]:
y_pred_trivial = np.zeros(len(y_test))
accuracy_trivial = (y_pred_trivial == y_test).mean()
print(f"Trivial classifier accuracy: {accuracy_trivial:.4%}")
print("Recall and precision are both 0 — model catches no fraud.")
print("This is why accuracy alone is useless on imbalanced problems.")

## Build a preprocessing pipeline

`type` is categorical → one-hot encode. Numeric features → scale. Use sklearn's `ColumnTransformer` so preprocessing is part of the model artifact (avoids train-serve skew at the simplest level).

In [ ]:
categorical_cols = ["type"]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ]
)

print(f"Numeric columns:    {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

## Model 1: Logistic regression

Linear model. Useful as a sanity check: if XGBoost only marginally beats this, we have feature-engineering work to do (or our 'fancy' model isn't actually learning much).

In [ ]:
with mlflow.start_run(run_name="logreg_baseline"):
    # Log hyperparameters
    mlflow.log_params(
        {
            "model_type": "logistic_regression",
            "max_iter": 1000,
            "class_weight": "balanced",
            "random_state": RANDOM_SEED,
        }
    )

    logreg = Pipeline(
        [
            ("prep", preprocessor),
            (
                "clf",
                LogisticRegression(
                    max_iter=1000,
                    class_weight="balanced",
                    random_state=RANDOM_SEED,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    logreg.fit(X_train, y_train)
    logreg_proba = logreg.predict_proba(X_test)[:, 1]
    logreg_metrics_default = evaluate_at_threshold(y_test.to_numpy(), logreg_proba, threshold=0.5)

    # Log metrics
    mlflow.log_metrics(
        {
            "precision": logreg_metrics_default.precision,
            "recall": logreg_metrics_default.recall,
            "f1": logreg_metrics_default.f1,
            "pr_auc": logreg_metrics_default.pr_auc,
            "roc_auc": logreg_metrics_default.roc_auc,
        }
    )

    # Log the model itself as an artifact
    mlflow.sklearn.log_model(logreg, artifact_path="model")

    print("Logistic regression at threshold=0.5:")
    print(logreg_metrics_default.pretty_print())

## Model 2: XGBoost

Tree-based, handles non-linearity, well-suited to tabular data. `scale_pos_weight` is XGBoost's class-weighting equivalent.

In [ ]:
# scale_pos_weight = ratio of negative to positive — counterbalances the imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {scale_pos_weight:.1f}")

with mlflow.start_run(run_name="xgboost_baseline"):
    # Log hyperparameters
    mlflow.log_params(
        {
            "model_type": "xgboost",
            "n_estimators": 200,
            "max_depth": 6,
            "learning_rate": 0.1,
            "scale_pos_weight": scale_pos_weight,
            "tree_method": "hist",
            "random_state": RANDOM_SEED,
        }
    )

    xgb_model = Pipeline(
        [
            ("prep", preprocessor),
            (
                "clf",
                xgb.XGBClassifier(
                    n_estimators=200,
                    max_depth=6,
                    learning_rate=0.1,
                    scale_pos_weight=scale_pos_weight,
                    random_state=RANDOM_SEED,
                    n_jobs=-1,
                    tree_method="hist",
                    eval_metric="aucpr",
                ),
            ),
        ]
    )
    xgb_model.fit(X_train, y_train)
    xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
    xgb_metrics_default = evaluate_at_threshold(y_test.to_numpy(), xgb_proba, threshold=0.5)

    # Calibrate threshold within the same run
    best_threshold = find_threshold_for_precision(y_test.to_numpy(), xgb_proba, min_precision=0.95)
    xgb_metrics_calibrated = evaluate_at_threshold(
        y_test.to_numpy(), xgb_proba, threshold=best_threshold
    )

    # Log both default-threshold and calibrated metrics with distinct names
    mlflow.log_metrics(
        {
            "precision_at_0.5": xgb_metrics_default.precision,
            "recall_at_0.5": xgb_metrics_default.recall,
            "f1_at_0.5": xgb_metrics_default.f1,
            "pr_auc": xgb_metrics_default.pr_auc,
            "roc_auc": xgb_metrics_default.roc_auc,
            "calibrated_threshold": best_threshold,
            "precision_calibrated": xgb_metrics_calibrated.precision,
            "recall_calibrated": xgb_metrics_calibrated.recall,
            "f1_calibrated": xgb_metrics_calibrated.f1,
        }
    )

    # Log model. mlflow.xgboost.log_model captures Booster directly,
    # but since we used a sklearn Pipeline wrapping XGBClassifier, use sklearn flavor.
    mlflow.sklearn.log_model(xgb_model, artifact_path="model")

    # Capture the active run ID — we'll need this if we want to register the model.
    active_run_id = mlflow.active_run().info.run_id
    print(f"\nXGBoost run logged with run_id={active_run_id}")
    print("\nXGBoost at threshold=0.5:")
    print(xgb_metrics_default.pretty_print())
    print(f"\nXGBoost at calibrated threshold={best_threshold:.4f}:")
    print(xgb_metrics_calibrated.pretty_print())

## Calibrate the threshold

Default threshold of 0.5 is rarely the business-optimal point. We require precision ≥ 0.95 (false positives are costly) and maximize recall under that constraint.

**Important:** threshold calibration must be done on data the model didn't train on, and it should ideally be a *separate* validation set — not the test set. For this notebook we'll use the test set for simplicity, but in production code we'd carve out a validation slice from the training data.

In [ ]:
best_threshold = find_threshold_for_precision(y_test.to_numpy(), xgb_proba, min_precision=0.95)
xgb_metrics_calibrated = evaluate_at_threshold(
    y_test.to_numpy(), xgb_proba, threshold=best_threshold
)
print(f"XGBoost at calibrated threshold={best_threshold:.4f}:")
print(xgb_metrics_calibrated.pretty_print())

## Plot precision-recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, proba in [("LogReg", logreg_proba), ("XGBoost", xgb_proba)]:
    p, r, _ = precision_recall_curve(y_test, proba)
    ax.plot(r, p, label=name)
ax.scatter(
    [xgb_metrics_calibrated.recall],
    [xgb_metrics_calibrated.precision],
    color="crimson",
    s=60,
    zorder=5,
    label=f"Operating point (t={best_threshold:.3f})",
)
ax.axhline(0.95, ls="--", color="gray", alpha=0.5, label="Precision floor 0.95")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-recall curve — baseline models")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "03_pr_curves.png", dpi=120)
plt.show()

In [ ]:
# Register the XGBoost calibrated model in the MLflow Model Registry.
#
# A "registered model" is a named, versioned model that lives independently
# of any single run. Multiple runs can produce versions of the same registered
# model; downstream code finds the right version via name + tag (e.g.,
# "give me the production version of fraud-detector").
#
# `active_run_id` was captured during the XGBoost training cell.
registered_model_name = "fraud-detector"

# The model URI points to the artifact within the run we want to register.
model_uri = f"runs:/{active_run_id}/model"

# This call (a) creates the registered model name on first run, and
# (b) adds a new version of it pointing at the artifact above.
model_version = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name,
)

print(f"✓ Registered '{registered_model_name}' version {model_version.version}")
print(f"  Source run: {active_run_id}")
print(f"  Status: {model_version.status}")

In [ ]:
# Promote this version to "production" using an alias.
#
# Aliases are pointers: "production" -> version 3 (e.g.). Reassigning the alias
# is how you do model rollouts and rollbacks atomically. No clients break;
# they all just resolve the alias to the new target.
#
# (MLflow used to use "stages" like Staging/Production, but these are
# deprecated in favor of aliases since MLflow 2.9.)
from mlflow.tracking import MlflowClient

client = MlflowClient()
client.set_registered_model_alias(
    name=registered_model_name,
    alias="production",
    version=model_version.version,
)

# Also log the calibrated threshold as a model-version tag — Lambda will
# need this alongside the model itself, and storing it with the model
# version keeps them aligned.
client.set_model_version_tag(
    name=registered_model_name,
    version=model_version.version,
    key="calibrated_threshold",
    value=str(best_threshold),
)
client.set_model_version_tag(
    name=registered_model_name,
    version=model_version.version,
    key="precision_at_calibration",
    value=str(xgb_metrics_calibrated.precision),
)
client.set_model_version_tag(
    name=registered_model_name,
    version=model_version.version,
    key="recall_at_calibration",
    value=str(xgb_metrics_calibrated.recall),
)

print(f"✓ Tagged version {model_version.version} as 'production' alias")
print(f"  Threshold tag: {best_threshold:.4f}")

## Persist the model and metrics

Two files written:
1. The model pipeline (preprocessing + classifier + threshold) as a single `.joblib`
2. A JSON of metrics for the next notebook to load and beat

Saving the threshold *with* the model is important — if you save them separately, eventually they'll drift apart.

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = f"baseline_xgboost_{timestamp}.joblib"
model_path = MODELS_DIR / model_filename

model_artifact = {
    "pipeline": xgb_model,
    "threshold": best_threshold,
    "metadata": {
        "created_at": timestamp,
        "random_seed": RANDOM_SEED,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "feature_columns": list(X_train.columns),
        "model_type": "xgboost_baseline",
    },
}
joblib.dump(model_artifact, model_path)
print(f"✓ Model saved to {model_path}")

# Update the 'latest' pointer so other code can find the most recent baseline.
latest_path = MODELS_DIR / "baseline_latest.joblib"
joblib.dump(model_artifact, latest_path)
print(f"✓ Latest baseline pointer: {latest_path}")

# Save metrics as JSON for diffing later.
metrics_path = MODELS_DIR / f"baseline_xgboost_{timestamp}_metrics.json"
metrics_path.write_text(
    json.dumps(
        {
            "logreg_default": logreg_metrics_default.to_dict(),
            "xgboost_default": xgb_metrics_default.to_dict(),
            "xgboost_calibrated": xgb_metrics_calibrated.to_dict(),
        },
        indent=2,
    )
)
print(f"✓ Metrics saved to {metrics_path}")

## Sanity-check: can we reload and reuse the model?

If this assertion fails, your saved artifact is broken — don't trust the rest of the pipeline.

In [ ]:
loaded = joblib.load(latest_path)
loaded_proba = loaded["pipeline"].predict_proba(X_test.head(100))[:, 1]
original_proba = xgb_model.predict_proba(X_test.head(100))[:, 1]

assert np.allclose(loaded_proba, original_proba), "Loaded model gives different predictions!"
print("✓ Model reload sanity check passed.")
print(f"✓ Loaded threshold: {loaded['threshold']:.4f}")

## What you have now

1. A trained XGBoost model with calibrated threshold, saved to `models/baseline_latest.joblib`.
2. Baseline metrics on the test set written to `models/*_metrics.json`.
3. A precision-recall curve in `reports/figures/`.

**Next:** notebook `03_feature_engineering.ipynb` — engineer time-windowed features and beat this baseline. The metrics JSON is your reference point.

In [ ]:
# Pretend we're Lambda. Load the production model purely by name+alias —
# no run IDs, no file paths, no joblib.
import mlflow.sklearn
from mlflow.tracking import MlflowClient

loaded_model = mlflow.sklearn.load_model("models:/fraud-detector@production")

# Directly resolve the alias to its model version — much more reliable
# than scanning all versions and filtering.
client = MlflowClient()
prod_version = client.get_model_version_by_alias(
    name="fraud-detector",
    alias="production",
)
production_threshold = float(prod_version.tags["calibrated_threshold"])

print("✓ Loaded model from alias 'production'")
print(f"  Resolved to version {prod_version.version}")
print(f"  Threshold: {production_threshold:.4f}")

# Run a prediction to prove it works end-to-end.
sample_proba = loaded_model.predict_proba(X_test.head(5))[:, 1]
sample_predictions = (sample_proba >= production_threshold).astype(int)
print(f"\n  Sample predictions: {sample_predictions.tolist()}")
print(f"  Sample probabilities: {sample_proba.round(4).tolist()}")

In [ ]:
# Find actual fraud examples in the test set and predict on them
fraud_indices = y_test[y_test == 1].index[:5]
fraud_samples = X_test.loc[fraud_indices]
fraud_proba = loaded_model.predict_proba(fraud_samples)[:, 1]
fraud_predictions = (fraud_proba >= production_threshold).astype(int)
print(f"Real-fraud probabilities: {fraud_proba.round(4).tolist()}")
print(f"Predictions on real fraud: {fraud_predictions.tolist()}")